# 03 · RQ2: early warning, lead time and haze episodes

Warning rule yhat > theta - delta (delta chosen on validation by F2, 6 h cooldown), hourly precision/recall/PR-AUC, event recall and horizon-matched lead time, regional haze episodes. Run notebook 02 first (it creates the models). The TEST split is evaluated once in a full run; quick runs are demos.

*Estimated time: QUICK: 1-2 min.*  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/furkantopcuu/airq-iot/blob/main/notebooks/03_early_warning_rq2.ipynb)

In [ ]:
# --- parameters -------------------------------------------------------------------------
QUICK = True        # 2 stations, short run (< 5 min); set False for the full 12-station run
USE_DRIVE = False   # Colab only: mount Google Drive to keep results between sessions
RUN_OPTUNA = False  # hyper-parameter search (notebook 02 onwards)
SEEDS = [0]


In [ ]:
# --- setup (runs on Colab, skipped locally) ---------------------------------------------
import os
import subprocess
import sys
IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/furkantopcuu/airq-iot.git"
if IN_COLAB:
    if not os.path.exists("/content/airq-iot"):
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/airq-iot"], check=True)
    os.chdir("/content/airq-iot")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]"], check=True)
    sys.path.insert(0, "/content/airq-iot/src")  # no restart needed after the editable install


In [ ]:
from airq.config import load_config
from airq.env import mount_drive_if_wanted
from airq.pipelines.colab import env_summary

mount_drive_if_wanted(USE_DRIVE)
cfg = load_config()
env_summary(cfg)


In [ ]:
from airq.pipelines.rq2 import run_rq2

summary = run_rq2(cfg, quick=QUICK, force=QUICK)  # full run: guarded, test is seen once
summary["n_test_exceedance_events"]


In [ ]:
import pandas as pd
from IPython.display import Image, display
from airq.env import resolve_paths
from airq.pipelines.figures import rq2

paths = resolve_paths(cfg["paths"])
tables = paths["reports"] / "tables"
w = pd.read_csv(tables / "rq2_warning.csv")
cols = ["grid", "method", "theta", "h", "delta", "precision", "recall", "f2", "pr_auc",
        "m_event_recall", "m_lead_median", "false_alarm_rate"]
display(w[w["grid"] == "spec"][cols].round(3))
display(pd.read_csv(tables / "rq2_haze.csv").round(3))
for p in rq2(cfg):
    if p.suffix == ".png":
        display(Image(filename=str(p)))


In [ ]:
# --- export: reports/ and models/ -> airq_artifacts_<date>.zip --------------------------
from airq.pipelines.colab import export_artifacts

zip_path = export_artifacts(cfg)
print("Wrote", zip_path.name)
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
# locally:  python -m scripts.import_artifacts <zip>   (also runs the manifest check)
